In [1]:
# Funciones auxiliares — Caso 1: Árbol de regresión (Car Price)
# Son las mismas que usamos en el laboratorio de árboles (caso MPG).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_regresion ahora también reporta el MAPE (%).
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE, MAE y MAPE (%) de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | "
          f"MAE {metricas['MAE']} | MAPE {metricas['MAPE (%)']}%")
    return metricas


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)

In [2]:
# Funciones auxiliares — Caso 2: Regresión logística (Phone Classification)
# Son las mismas que usamos en los laboratorios de regresión logística y árboles.
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_clasificador y plot_matriz_confusion se ajustaron para funcionar
#      con MÁS DE DOS clases (en Titanic sólo había dos: sobrevivió / no sobrevivió).
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_rfecv_clasificacion(rfecv, metrica='exactitud'):
    """Desempeño promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name=f'{metrica} promedio (CV)'
    ))
    fig.add_vline(x=rfecv.n_features_, line_dash='dash', line_color='red',
                  annotation_text=f'elegido: {rfecv.n_features_}')
    fig.update_layout(
        title='RFECV — desempeño según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title=f'{metrica.capitalize()} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1.

    Con más de dos clases, precisión, recall y F1 son el promedio simple
    de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos,
            p. ej. ['Low', 'Mid_Low', 'Mid', 'High']. Si no las pasas, se ordenan alfabéticamente.
    etiquetas: nombres a mostrar para esas clases (opcional; por defecto, las mismas clases).
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm

In [3]:
tp = 11
fp = 1043
precision = tp / (tp + fp)
print(f"Precision: {precision:.2f}")

Precision: 0.01


In [4]:
tp1 = 1621
fp1 = 490
fn1 = 731
recall = tp1 / (tp1 + fn1)
print(f"Recall: {recall:.2f}")

Recall: 0.69


Pregunta 11

In [5]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/CarPrice_dataset.csv')

# Mostrar las primeras filas
print("Muestra del dataset:")
df.head()

Muestra del dataset:


,CarName,carbody,fueltype,enginetype,cylindernumber,doornumber,symboling,wheelbase,carlength,carwidth,...,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,alfa-romero giulia,convertible,gas,dohc,4,2,3,88.6,168.8,64.1,...,2548,130,3.47,2.68,9.0,111,5000,21,27,13495.0
1,alfa-romero stelvio,convertible,gas,dohc,4,2,3,88.6,168.8,64.1,...,2548,130,3.47,2.68,9.0,111,5000,21,27,16500.0
2,alfa-romero Quadrifoglio,hatchback,gas,ohcv,6,2,1,94.5,171.2,65.5,...,2823,152,2.68,3.47,9.0,154,5000,19,26,16500.0
3,audi 100 ls,sedan,gas,ohc,4,4,2,99.8,176.6,66.2,...,2337,109,3.19,3.40,10.0,102,5500,24,30,13950.0
4,audi 100ls,sedan,gas,ohc,5,4,2,99.4,176.6,66.4,...,2824,136,3.19,3.40,8.0,115,5500,18,22,17450.0


In [6]:
df.sum()

CarName             alfa-romero giuliaalfa-romero stelvioalfa-rome...
carbody             convertibleconvertiblehatchbacksedansedansedan...
fueltype            gasgasgasgasgasgasgasgasgasgasgasgasgasgasgasg...
enginetype          dohcdohcohcvohcohcohcohcohcohcohcohcohcohcohco...
cylindernumber                                                    898
doornumber                                                        640
symboling                                                         171
wheelbase                                                     20245.1
carlength                                                     35680.1
carwidth                                                      13511.1
carheight                                                     11013.6
curbweight                                                     523891
enginesize                                                      26016
boreratio                                                       682.6
stroke              

In [7]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['price', 'CarName', 'carbody', 'fueltype', 'enginetype'])
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [8]:
from sklearn.tree import DecisionTreeRegressor
arbol_sin_podar = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
comparacion = []
comparacion.append(
    evaluar_regresion(
        'Árbol sin podar', 
        y_test, 
        arbol_sin_podar.predict(X_test)
    
        )
)
print("R2 en train", arbol_sin_podar.score(X_train, y_train))
print("Numero hojas", arbol_sin_podar.get_n_leaves())
print("Profundidad", arbol_sin_podar.get_depth())

Árbol sin podar: R² 0.8589 | RMSE 3304.8039 | MAE 2169.317 | MAPE 17.0%
R2 en train 0.9984642634434308
Numero hojas 150
Profundidad 14


In [9]:
#Escribir las importancias del modelo según el atributo feature_importances
importancias = pd.Series(
    arbol_sin_podar.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Importancias porcentuales:")
print((importancias).round(4))

Importancias porcentuales:
enginesize          0.6754
curbweight          0.2356
carwidth            0.0197
compressionratio    0.0164
peakrpm             0.0158
horsepower          0.0117
stroke              0.0069
carlength           0.0059
symboling           0.0041
citympg             0.0034
wheelbase           0.0022
highwaympg          0.0011
boreratio           0.0008
carheight           0.0007
doornumber          0.0004
cylindernumber      0.0001
dtype: float64


In [10]:
#Encontrar la importancia acumulada que cruzan el .80
importancia_acumulada = importancias.cumsum()
print(importancia_acumulada)

enginesize          0.675420
curbweight          0.910998
carwidth            0.930696
compressionratio    0.947097
peakrpm             0.962876
horsepower          0.974532
stroke              0.981415
carlength           0.987272
symboling           0.991339
citympg             0.994751
wheelbase           0.996963
highwaympg          0.998110
boreratio           0.998880
carheight           0.999550
doornumber          0.999927
cylindernumber      1.000000
dtype: float64


In [11]:
#Nueva base de datos que contiene sólo las variables que cruzan el .80 de importancia acumulada y la variable objetivo
variables_seleccionadas = importancia_acumulada[importancia_acumulada <= 0.80].index.tolist()
variables_seleccionadas.append('price')  # Agregar la variable objetivo
df_reducido = df[variables_seleccionadas]

In [12]:
from sklearn.model_selection import train_test_split
X = df_reducido.drop(columns=['price'])
y = df_reducido['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [13]:
arbol_sin_podar = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
comparacion = []
comparacion.append(
    evaluar_regresion(
        'Árbol sin podar', 
        y_test, 
        arbol_sin_podar.predict(X_test)
    
        )
)
print("R2 en train", arbol_sin_podar.score(X_train, y_train))
print("Numero hojas", arbol_sin_podar.get_n_leaves())
print("Profundidad", arbol_sin_podar.get_depth())

Árbol sin podar: R² 0.8092 | RMSE 3843.3061 | MAE 2633.765 | MAPE 24.63%
R2 en train 0.938342937929238
Numero hojas 38
Profundidad 7


In [14]:
X_train.shape

(164, 1)

In [15]:
X_test.shape

(41, 1)

In [16]:
y_train.shape

(164,)

In [17]:
y_test.shape

(41,)

In [18]:
#R2 de train - R2 de test
dif = arbol_sin_podar.score(X_train, y_train) - arbol_sin_podar.score(X_test, y_test)
print("Diferencia R2:", dif)

Diferencia R2: 0.1291424342928934


In [19]:
#Obtener MAPE  de train y test
mape_train = mean_absolute_percentage_error(y_train, arbol_sin_podar.predict(X_train)) * 100
mape_test = mean_absolute_percentage_error(y_test, arbol_sin_podar.predict(X_test)) * 100
print(f"MAPE de train: {mape_train:.2f}%")
print(f"MAPE de test: {mape_test:.2f}%")

MAPE de train: 11.47%
MAPE de test: 24.63%


In [20]:
#Obtener RMSE de train y test
rmse_train = np.sqrt(mean_squared_error(y_train, arbol_sin_podar.predict(X_train)))
rmse_test = np.sqrt(mean_squared_error(y_test, arbol_sin_podar.predict(X_test)))
print(f"RMSE de train: {rmse_train:.2f}") 
print(f"RMSE de test: {rmse_test:.2f}")    

RMSE de train: 1923.87
RMSE de test: 3843.31


In [21]:
avg_price = df['price'].mean().round(2)
mae_test = mean_absolute_error(y_test, arbol_sin_podar.predict(X_test))
percent_error = (mae_test / avg_price) * 100
print(f'error porcentual: {percent_error:.2f}%')

error porcentual: 19.84%


Caso 2

In [22]:
import pandas as pd

url = "https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/phone_classification.csv"
df = pd.read_csv(url)
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,px_height,px_width,ram,talk_time,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,20,756,2549,19,0,1,Mid_Low
1,1021,1,0.5,1,0,1,53,0.7,136,3,905,1988,2631,7,1,0,Mid
2,563,1,0.5,1,2,1,41,0.9,145,5,1263,1716,2603,9,1,0,Mid
3,615,1,2.5,0,0,0,10,0.8,131,6,1216,1786,2769,11,0,0,Mid
4,1821,1,1.2,0,13,1,44,0.6,141,2,1208,1212,1411,15,1,0,Mid_Low


In [23]:
#genera el gráfico de barras cantidad de observaciones por price_range con plotly
import plotly.express as px

fig = px.histogram(df, x='price_range', title='Distribución de Precios')
fig.show()

In [24]:
#Relaciones lineales entre las variables independientes y price_range
orden = {"Low": 0, "Mid_Low": 1, "Mid": 2, "High": 3}
y = df["price_range"].map(orden)

correlaciones = (
    df.select_dtypes(include="number")
      .drop(columns="price_range", errors="ignore")
      .corrwith(y)
)

print(correlaciones.abs().sort_values(ascending=False))
print("Mayor relación:", correlaciones.abs().idxmax(), correlaciones.abs().max())
print("Menor relación:", correlaciones.abs().idxmin(), correlaciones.abs().min())


ram              0.921647
battery_power    0.206876
px_width         0.171549
px_height        0.156043
int_memory       0.047408
blue             0.031410
mobile_wt        0.030850
touch_screen     0.029865
fc               0.029378
talk_time        0.025332
wifi             0.021363
dual_sim         0.016497
four_g           0.015491
m_dep            0.004895
n_cores          0.002594
clock_speed      0.002126
dtype: float64
Mayor relación: ram 0.9216471907882475
Menor relación: clock_speed 0.002126495749355456


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1850 entries, 0 to 1849
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   battery_power  1850 non-null   int64  
 1   blue           1850 non-null   int64  
 2   clock_speed    1850 non-null   float64
 3   dual_sim       1850 non-null   int64  
 4   fc             1850 non-null   int64  
 5   four_g         1850 non-null   int64  
 6   int_memory     1850 non-null   int64  
 7   m_dep          1850 non-null   float64
 8   mobile_wt      1850 non-null   int64  
 9   n_cores        1850 non-null   int64  
 10  px_height      1850 non-null   int64  
 11  px_width       1850 non-null   int64  
 12  ram            1850 non-null   int64  
 13  talk_time      1850 non-null   int64  
 14  touch_screen   1850 non-null   int64  
 15  wifi           1850 non-null   int64  
 16  price_range    1850 non-null   str    
dtypes: float64(2), int64(14), str(1)
memory usage: 245.8 KB


In [26]:
X = df.drop(columns=['price_range'])
y = df['price_range']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [27]:
#Escala las variables independientes con StandardScaler, ajustándolo sólo con X_train.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [31]:
#Ajusta una Regresión Logística (random_state=0, max_iter=500) con sklearn y realiza la eliminación recursiva de características dentro de un ciclo de validación cruzada (RFECV, cv=5) sobre X_train escalado.
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

dat = RFECV(
    estimator=LogisticRegression(random_state=0, max_iter=500),
    cv=5,
)

dat.fit(X_train_scaled, y_train)
print(f"Mejor número de variables: {dat.n_features_}")

Mejor número de variables: 5


In [32]:
# ¿Cuáles fueron las variables seleccionadas por RFECV (X.columns[rfecv.support_])?
X.columns[dat.support_]

Index(['battery_power', 'mobile_wt', 'px_height', 'px_width', 'ram'], dtype='str')

In [41]:
# matriz de correlacion
dat_features = X.columns[dat.support_]
plot_correlation_matrix(df, dat_features)


In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

dat = RFECV(
    estimator=LogisticRegression(random_state=0, max_iter=500),
    cv=5,
)

dat.fit(X_train_scaled, y_train)
print(f"Mejor número de variables: {}")

Mejor número de variables: [ True False False False False False False False  True False  True  True
  True False False False]


In [44]:
from sklearn.linear_model import LogisticRegression

X_train_selected = X_train_scaled[:, dat.support_]
X_test_selected = X_test_scaled[:, dat.support_]

model = LogisticRegression(random_state=0, max_iter=500)
model.fit(X_train_selected, y_train)

score = model.score(X_test_selected, y_test)
print("Accuracy:", score)


NameError: name 'X_test_scaled' is not defined